# On-Policy Knowledge Distillation for Structured Tool-Use Syntax

### 1. Project Objective & Scope
Implement a lightweight, end-to-end, **On-Policy Knowledge Distillation** pipeline from scratch using PyTorch. The target task is **Structured Tool-Use Formatting** (specifically, a Calculator tool pattern for math reasoning). 

Instead of training a student model on a static dataset of perfect text (Off-Policy/SFT), the student model will generate its own responses token-by-token (On-Policy Rollouts). A frozen teacher model will evaluate these exact trajectories in real-time, providing dense token-level feedback via **Reverse Kullback-Leibler (KL) Divergence** to correct the student's distribution and eliminate exposure bias.

### 2. System Architecture & Model Selection
To accommodate local hardware constraints (4GB Dedicated VRAM), we employ a quantized teacher-student paradigm:

*   **Student Policy ($\pi_\theta$):** `Qwen/Qwen2.5-0.5B-Instruct` (Loaded in BF16/FP16 ~0.95 GB VRAM). This model is highly efficient but structurally chaotic at tool-formatting without alignment.
*   **Teacher Policy ($\pi_{\text{teacher}}$):** `Qwen/Qwen2.5-1.5B-Instruct` (Quantized to 4-bit via bitsandbytes ~1.10 GB VRAM). This serves as our frozen anchor distribution.

### 3. Mathematical Formulation
For every prompt $x$ sampled from synthetic dataset, the training loop executes four phases:

#### _Phase 1: On-Policy Rollout Generation_
The student model auto-regressively samples a token sequence (trajectory) $\tau$ consisting of actions (tokens) $a_t$ conditioned on the evolving state $s_t$:
$$\tau \sim \pi_\theta(a_t \mid s_t)$$

#### _Phase 2: Dual Logit Extraction_
The generated trajectory $\tau$ (by student) is fed back into both models (teacher and student) simultaneously with gradients disabled for the teacher. We extract the raw logit vectors for every token step:
$$z^{\text{student}}_t, z^{\text{teacher}}_t = \text{Forward}(\tau)$$

#### _Phase 3: Token-Level Reverse KL Divergence_
Apply the Softmax and Log-Softmax functions to calculate the Reverse KL Divergence token-by-token across the trajectory (at each time-step $t$:)
$$\mathbb{D}_{\text{KL}}\left(\pi_\theta(\cdot \mid s_t) \parallel \pi_{\text{teacher}}(\cdot \mid s_t)\right) = \sum_{w \in \mathcal{V}} \pi_\theta(w \mid s_t) \cdot \left[ \log \pi_\theta(w \mid s_t) - \log \pi_{\text{teacher}}(w \mid s_t) \right]$$

#### _Phase 4: Parameter Optimization_
The student's internal weights ($\theta$) are optimized by minimizing the accumulated trajectory divergence We run loss.backward() and update the Student's parameters (θ) using an AdamW optimizer:
$$\mathcal{L}_{\text{distill}}(\theta) = \frac{1}{T}\sum_{t=1}^T \mathbb{D}_{\text{KL}}\left(\pi_\theta(\cdot \mid s_t) \parallel \pi_{\text{teacher}}(\cdot \mid s_t)\right)$$

In [1]:
# 1 Environment Setup
import sys
import subprocess
import re
import os

import bitsandbytes as bnb
import torch
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Random Seeds for Reproducibility 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"• PyTorch Version: {torch.__version__}")
print(f"• GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"• VRAM: {torch.cuda.get_device_name(0)}")


• PyTorch Version: 2.9.0+cu130
• GPU Available: True
• VRAM: NVIDIA GeForce GTX 1650


#### **Methodological Choice: Synthetic Prompt (data) Generation vs. Benchmark Datasets**

In this isolation experiments, I'm utilizing controlled **synthetic datasets** which is often superior to using static public benchmarks like GSM8K. This choice provides three core advantages:

1. **Isolation of Variables:** Public datasets contain complex linguistic variation, diverse formatting conventions, and varying degrees of difficulty. By using a programmatic generator, we eliminate text noise and ensure that any changes in model performance are directly caused by our **On-Policy Distillation Algorithm**, not data unpredictability.
2. **Deterministic Task Alignment:** The target of this experiment is to train a model to output a strict `[CALCULATOR(expression)]` tool sequence. Synthetic generation guarantees that *100%* of our data entries require this exact tool behavior, creating a perfectly clean optimization path.
3. **Reproducibility & Scale:** We can scale the dataset size instantly from 128 samples to 10,000 samples without dealing with external file dependencies, corrupted lines, or downloading multi-gigabyte data files on consumer hardware.


In [2]:
# 2: Synthetic Data Generation
class MathPromptGenerator(Dataset):
    """
    A synthetic data generator that creates arithmetic prompts.
    This mimics a 'Reader' in a professional pipeline.
    """
    def __init__(self, num_samples=100, tokenizer=None):
        self.samples = []
        self.tokenizer = tokenizer
        self.operations = ["add", "multiply", "subtract"]
        
        print(f"• Generating {num_samples} synthetic math prompts...")
        
        for _ in range(num_samples):
            # Generate random numbers for simple arithmetic
            a = random.randint(10, 999)
            b = random.randint(10, 999)
            op = random.choice(self.operations)
            
            if op == "add":
                question = f"What is {a} plus {b}?"
            elif op == "multiply":
                question = f"Calculate {a} * {b}."
            elif op == "subtract":
                question = f"What is {a} minus {b}?"
            
            # Formatting: Apply the Official Qwen Chat Template
            # We only provide the USER role. The model must generate the ASSISTANT role.
            messages = [
                {"role": "system", "content": "You are a helpful math assistant. Use the tool [CALCULATOR(expression)] to solve problems."},
                {"role": "user", "content": question}
            ]
            
            # Apply chat template but DO NOT generate the assistant response yet
            formatted_prompt = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            self.samples.append(formatted_prompt)
            
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# We need a tokenizer to format the prompts correctly, so we load it briefly here
# (We will reload it with the model in the next cell, but this keeps data logic separate)
print("• Loading Tokenizer for Data Formatting...")
tokenizer_loader = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", trust_remote_code=True)

# Create the Dataset
train_dataset = MathPromptGenerator(num_samples=128, tokenizer=tokenizer_loader)

# Preview a sample to verify the format
print("\nSample Data Entry (What the Student sees):")
print("-" * 50)
print(train_dataset[0])
print("-" * 50)


• Loading Tokenizer for Data Formatting...
• Generating 128 synthetic math prompts...

Sample Data Entry (What the Student sees):
--------------------------------------------------
<|im_start|>system
You are a helpful math assistant. Use the tool [CALCULATOR(expression)] to solve problems.<|im_end|>
<|im_start|>user
What is 664 plus 124?<|im_end|>
<|im_start|>assistant

--------------------------------------------------


#### Quantized Teacher & Student Model Loading
I use `BitsAndBytesConfig` to enforce 4-bit quantization on the Teacher model. To keeps the entire training script running within physical 4GB Dedicated VRAM without crashing.

In [3]:
# 3 Memory-Optimized Model Initialization

STUDENT_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TEACHER_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit Quantization for the Teacher
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("↻ Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)

# Enforce left padding for decoder-only architectures
tokenizer.padding_side = "left"

print("↻ Loading Teacher Model in 4-bit...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
teacher_model.eval()

print("↻ Loading Student Model in 16-bit...")
# load the student in bfloat16/float16 for active training
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
student_model.train() # set student to training mode

print("\n• VRAM Optimization Status:")
print(f"    - Teacher Device: {next(teacher_model.parameters()).device}")
print(f"    - Student Device: {next(student_model.parameters()).device}")
print("✓ Both models loaded successfully within memory bounds!")


#verifying EOS
print(tokenizer.pad_token, tokenizer.pad_token_id)   
print(tokenizer.eos_token, tokenizer.eos_token_id)  
assert tokenizer.pad_token_id != tokenizer.eos_token_id

↻ Loading Tokenizer...
↻ Loading Teacher Model in 4-bit...


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu130).
W0518 19:54:14.757000 8436 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


↻ Loading Student Model in 16-bit...


`torch_dtype` is deprecated! Use `dtype` instead!



• VRAM Optimization Status:
    - Teacher Device: cuda:0
    - Student Device: cuda:0
✓ Both models loaded successfully within memory bounds!
<|endoftext|> 151643
<|im_end|> 151645


In [4]:
# 4: Hyperparameters config

OPTIMIZER = bnb.optim.PagedAdamW8bit(student_model.parameters(), lr=5e-6)
BATCH_SIZE = 2
EPOCHS = 1

#### On-Policy Distillation Training Loop
The distillation engine, it rolls out responses from the student, fetches parallel predictions, extracts log-probabilities, and calculates the Reverse KL Divergence Loss token-by-token.

In [5]:
# 5: On-Policy Distillation Engine

# Create DataLoader for prompt management
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("↻ Launching Stabilised On-Policy Distillation Loop...")

for epoch in range(EPOCHS):
    total_epoch_loss = 0.0
    
    for step, batch_prompts in enumerate(train_loader):
        OPTIMIZER.zero_grad()
        
        # Tokenize incoming prompts with explicit tensor targets
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to("cuda")
        prompt_attention_mask = inputs["attention_mask"].to("cuda")
        
        prompt_len = input_ids.shape[1]
        
        # --- PHASE 1: ON-POLICY ROLLOUT GENERATION ---
        with torch.no_grad():
            rollout_outputs = student_model.generate(
                input_ids=input_ids,
                attention_mask=prompt_attention_mask,
                max_new_tokens=32,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Extract the complete trajectory matrix
        trajectory_ids = rollout_outputs
        trajectory_attention_mask = trajectory_ids.ne(tokenizer.pad_token_id).long()
        
        # --- PHASE 2: DUAL LOGIT EXTRACTION ---
        student_outputs = student_model(trajectory_ids, attention_mask=trajectory_attention_mask)
        student_logits = student_outputs.logits
        
        with torch.no_grad():
            teacher_outputs = teacher_model(trajectory_ids, attention_mask=trajectory_attention_mask)
            teacher_logits = teacher_outputs.logits
            
        # --- PHASE 3: DYNAMIC REVERSE KL COMPUTATION ---
        # Shift logits by 1 for next-token auto-regressive alignment
        shift_student_logits = student_logits[:, :-1, :].contiguous()
        shift_teacher_logits = teacher_logits[:, :-1, :].contiguous()
        
        # Calculate full vocabulary distributions safely
        log_p_student = F.log_softmax(shift_student_logits, dim=-1)
        p_student = F.softmax(shift_student_logits, dim=-1)
        log_p_teacher = F.log_softmax(shift_teacher_logits, dim=-1)
        
        # CRITICAL STABILITY FIX: Clamp log-probs to prevent -inf subtraction blowups
        log_p_student = torch.clamp(log_p_student, min=-50.0, max=0.0)
        log_p_teacher = torch.clamp(log_p_teacher, min=-50.0, max=0.0)
        
        # Compute Reverse KL element-wise safely
        kl_per_token = p_student * (log_p_student - log_p_teacher)
        kl_sum = kl_per_token.sum(dim=-1)
        
        # Isolate loss exclusively to the generated tokens
        loss_mask = torch.zeros_like(trajectory_ids[:, :-1], dtype=torch.float32)
        loss_mask[:, prompt_len-1:] = 1.0
        final_mask = loss_mask * trajectory_attention_mask[:, :-1]
        
        # Apply the tracking mask and compute the true sequence average
        masked_loss = (kl_sum * final_mask).sum() / final_mask.sum().clamp(min=1.0)
        
        # Safety Check: If something goes wrong, skip backward pass to prevent hardware crash
        if torch.isnan(masked_loss) or torch.isinf(masked_loss):
            print(f"WARNING: Skipped unstable batch {step} due to mathematical infinity detect.")
            continue
            
        # --- PHASE 4: PARAMETER OPTIMIZATION ---
        masked_loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
        OPTIMIZER.step()

        bad = [n for n, p in student_model.named_parameters() if not torch.isfinite(p).all()]
        if bad:
            print(f"NaN/Inf in {len(bad)} params after step {step}, first: {bad[0]}")
            break
        
        total_epoch_loss += masked_loss.item()
        
        if step % 10 == 0:
            print(f"Batch {step}/{len(train_loader)} | Running Stable Loss: {masked_loss.item():.4f}")
   
    print(f"\n * Epoch {epoch+1} Complete. Stable Average Loss: {total_epoch_loss / len(train_loader):.4f}\n")


↻ Launching Stabilised On-Policy Distillation Loop...
Batch 0/64 | Running Stable Loss: 0.2064
Batch 10/64 | Running Stable Loss: 0.2440
Batch 20/64 | Running Stable Loss: 0.1426
Batch 30/64 | Running Stable Loss: 0.0397
Batch 40/64 | Running Stable Loss: 0.0171
Batch 50/64 | Running Stable Loss: 0.0391
Batch 60/64 | Running Stable Loss: 0.0412

 * Epoch 1 Complete. Stable Average Loss: 0.0758



In [6]:
# 6: Comparative Inference Evaluation

# Switch the student model to evaluation mode (disables dropout/gradients)
student_model.eval()

# 1. Define a fresh test prompt the model didn't see during training
test_question = "What is 45 plus 15?"

# 2. Apply the official chat structure template
test_messages = [
    {"role": "system", "content": "You are a helpful math assistant. Use the tool [CALCULATOR(expression)] to solve problems."},
    {"role": "user", "content": test_question}
]
formatted_test_prompt = tokenizer.apply_chat_template(
    test_messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# 3. Tokenize the input string
eval_inputs = tokenizer([formatted_test_prompt], return_tensors="pt").to("cuda")

print("[] Evaluating Aligned Student Generation...")
print("-" * 60)

# 4. Generate the response
with torch.no_grad():
    generated_ids = student_model.generate(
        **eval_inputs,
        max_new_tokens=48,
        do_sample=False, # Use greedy decoding to see its absolute strongest belief
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# 5. Decode the raw tokens back into human-readable text
# We only want to print what the model added *after* the input prompt
prompt_length = eval_inputs.input_ids.shape[1]
generated_tokens = generated_ids[0][prompt_length:]
output_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f"Prompt: {test_question}")
print(f"Generated Output:\n{output_text.strip()}")
print("-" * 60)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[] Evaluating Aligned Student Generation...
------------------------------------------------------------
Prompt: What is 45 plus 15?
Generated Output:
To find the sum of 45 and 15, you simply add them together:

\[ 45 + 15 = 60 \]

So, 45 plus 15 equals 60.
------------------------------------------------------------


Feeds a batch of test prompts to the model and calculates the exact percentage of times it followed the strict [CALCULATOR(...)] formatting rule perfectly.

In [7]:
# 7: Quantitative Metric Evaluation

student_model.eval()

# Generate a small evaluation batch (10 unseen prompts)
eval_questions = [
    f"What is {random.randint(10,99)} plus {random.randint(10,99)}?" for _ in range(10)
]
success_count = 0

print("Testing 10 distinct prompts for format compliance:..\n")

for i, q in enumerate(eval_questions):
    msg = [
        {"role": "system", "content": "You are a helpful math assistant. Use the tool [CALCULATOR(expression)] to solve problems."},
        {"role": "user", "content": q}
    ]
    fmt_p = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([fmt_p], return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        gen_ids = student_model.generate(
            **inputs, 
            max_new_tokens=32, 
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Isolate generated tokens
    prompt_len = inputs.input_ids.shape[1]
    output = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True).strip()
    
    # Regex Check: Does it contain '[CALCULATOR(' followed by numbers/operators and a closing ']'?
    pattern = r"\[CALCULATOR\([\d\+\-\*\/ ]+\)\]"
    is_valid = bool(re.search(pattern, output))
    
    if is_valid:
        success_count += 1
        status = "(✓) PASSED"
    else:
        status = "✗) FAILED"
        
    print(f"Sample {i+1} | {status} | Output: {output}")

success_rate = (success_count / len(eval_questions)) * 100
print(f"\n★ Final Quantitative Formatting Success Rate: {success_rate:.1f}%")


Testing 10 distinct prompts for format compliance:..

Sample 1 | ✗) FAILED | Output: To find the sum of 81 and 94, you simply add them together:

\[ 81 + 94 = 175
Sample 2 | ✗) FAILED | Output: To find the sum of 72 and 29, you simply add them together:

\[ 72 + 29 = 101
Sample 3 | ✗) FAILED | Output: To find the sum of 34 and 47, you simply add them together:

\[ 34 + 47 = 81 \
Sample 4 | ✗) FAILED | Output: To find the sum of 37 and 17, you simply add them together:

\[ 37 + 17 = 54 \
Sample 5 | ✗) FAILED | Output: To find the sum of 84 and 79, you simply add them together:

\[ 84 + 79 = 163
Sample 6 | ✗) FAILED | Output: To find the sum of 17 and 50, you simply add them together:

\[ 17 + 50 = 67 \
Sample 7 | ✗) FAILED | Output: To find the sum of 17 and 16, you simply add them together:

\[ 17 + 16 = 33 \
Sample 8 | ✗) FAILED | Output: To find the sum of 84 and 71, you simply add them together:

\[ 84 + 71 = 155
Sample 9 | ✗) FAILED | Output: To find the sum of 74 and 77, you simpl

In [ ]:
# 8: Save Aligned Student Weights
SAVE_DIRECTORY = "./on_policy_distilled_qwen"

print(f"Saving your trained Student model to: {SAVE_DIRECTORY} ...")

# Create directory if it doesn't exist
os.makedirs(SAVE_DIRECTORY, exist_ok=True)

# Save the weights and tokenizer configurations
student_model.save_pretrained(SAVE_DIRECTORY)
tokenizer.save_pretrained(SAVE_DIRECTORY)

print("✓ Student model and tokenizer saved successfully!")

Saving your trained Student model to: ./on_policy_distilled_qwen ...
✓ Student model and tokenizer saved successfully!
